In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Skdeo@62",
    database="financial_analytics"
)

print("Connected:", conn.is_connected())

In [ ]:
import pandas as pd

def run_query(sql):
    return pd.read_sql(sql, conn)

In [ ]:
df = run_query("""
    SELECT *
    FROM companies
    LIMIT 10
""")

df

In [ ]:
def get_company_profile(company_name):
    sql = f"""
        SELECT *
        FROM company_master_profile
        WHERE company_name = '{company_name}'
           OR company_id = '{company_name.upper()}'
    """
    
    return run_query(sql)

In [ ]:
get_company_profile("TCS")

In [ ]:
def get_sector_analysis():
    sql = """
        SELECT *
        FROM sector_performance
        ORDER BY avg_market_cap_crore DESC
    """
    
    return run_query(sql)

In [ ]:
get_sector_analysis()

In [ ]:
def get_valuation():
    sql = """
        SELECT *
        FROM company_valuation
        ORDER BY pe_ratio ASC
    """
    
    return run_query(sql)

In [ ]:
get_valuation().head(10)

In [ ]:
def get_stock_performance():
    sql = """
        SELECT *
        FROM company_stock_performance
        ORDER BY return_pct DESC
    """
    
    return run_query(sql)

In [ ]:
get_stock_performance().head(10)

In [ ]:
def compare_companies(company1, company2):
    sql = f"""
        SELECT
            company_id,
            company_name,
            broad_sector,
            return_on_equity_pct,
            net_profit_margin_pct,
            debt_to_equity,
            pe_ratio,
            start_price,
            latest_price,
            return_pct
        FROM company_master_profile
        WHERE company_id IN ('{company1.upper()}', '{company2.upper()}')
    """

    return run_query(sql)

In [ ]:
compare_companies("TCS", "INFY")

In [ ]:
#SQL Safety Validator
import re #re is Python's Regular Expression module.

def validate_sql(sql):
    sql_clean = sql.strip().upper()

    # Must start with SELECT or WITH
    if not (sql_clean.startswith("SELECT") or sql_clean.startswith("WITH")):
        return False, "Only SELECT or WITH queries are allowed."

    # Block write/destructive operations
    forbidden = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "DROP",
        "ALTER",
        "TRUNCATE",
        "CREATE",
        "RENAME",
        "GRANT",
        "REVOKE"
    ]

    for keyword in forbidden:
        if re.search(rf"\b{keyword}\b", sql_clean):
            return False, f"Forbidden SQL operation: {keyword}"

    return True, "SQL is safe."

In [ ]:
print(validate_sql("SELECT * FROM companies LIMIT 5"))

In [ ]:
print(validate_sql("DROP TABLE companies"))

In [ ]:
#Safe SQL Execution
def run_safe_query(sql):
    is_safe, message = validate_sql(sql)

    if not is_safe:
        raise ValueError(message)

    return run_query(sql)


In [ ]:
result = run_safe_query("""
    SELECT id, company_name
    FROM companies
    LIMIT 5
""")

result

In [ ]:
#run_safe_query("DROP TABLE companies")

In [ ]:
#Get Database Schema

def get_schema():
    tables = run_query("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'financial_analytics'
        ORDER BY table_name
    """)

    schema = {}

    for table in tables["TABLE_NAME"]:
        columns = run_query(f"""
            DESCRIBE `{table}`
        """)

        schema[table] = columns[["Field", "Type"]].to_dict("records")

    return schema

In [ ]:
schema = get_schema()

list(schema.keys())

In [ ]:
#Convert Schema to LLM-Friendly Text
def schema_to_text(schema):
    lines = []

    for table, columns in schema.items():
        lines.append(f"\nTABLE: {table}")

        for column in columns:
            lines.append(
                f"  - {column['Field']} ({column['Type']})"
            )

    return "\n".join(lines)

In [ ]:
schema_text = schema_to_text(schema)

print(schema_text)

In [ ]:
SYSTEM_PROMPT = f"""
You are an AI-powered financial data analyst.

You have access to a MySQL database named `financial_analytics`.

Your job is to:
1. Understand the user's financial question.
2. Generate a valid MySQL query to answer it.
3. Use only the tables and columns provided in the database schema.
4. Generate READ-ONLY SQL queries only.
5. Never use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE, RENAME, GRANT, or REVOKE.
6. Prefer the analytical views when they can answer the question directly.
7. Return accurate, concise answers based on the database results.
8. Do not invent financial data.

DATABASE SCHEMA:
{schema_text}
"""

In [ ]:
print(SYSTEM_PROMPT[:2000])

In [ ]:
%pip install -U google-genai

In [ ]:
import google.genai

print("Gemini SDK:", google.genai.__version__)

In [ ]:
import os

api_key = os.getenv("GEMINI_API_KEY")

print("Gemini API key loaded:", api_key is not None)

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client created successfully.")

In [ ]:
response = client.interactions.create(
    model="gemini-3.6-flash",
    input="Say hello as an AI financial analyst."
)

print(response.output_text)

In [ ]:
response = client.interactions.create(
    model="gemini-3.6-flash",
    system_instruction=SYSTEM_PROMPT,
    input="What is the average ROE by sector?"
)

print(response.output_text)

In [ ]:
def generate_sql(question):
    prompt = f"""
Generate ONLY the MySQL query needed to answer the user's question.

Do not explain the query.
Do not use markdown.
Do not use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE, RENAME, GRANT, or REVOKE.

User question:
{question}
"""

    response = client.interactions.create(
        model="gemini-3.6-flash",
        system_instruction=SYSTEM_PROMPT,
        input=prompt
    )

    return response.output_text.strip()

In [ ]:
sql = generate_sql("What is the average ROE by sector?")

print(sql)

In [ ]:
def ask_database(question):
    # Step 1: Generate SQL with Gemini
    sql = generate_sql(question)

    print("Generated SQL:")
    print(sql)

    # Step 2: Validate SQL
    is_safe, message = validate_sql(sql)

    if not is_safe:
        raise ValueError(f"Unsafe SQL blocked: {message}")

    # Step 3: Execute safely
    result = run_safe_query(sql)

    return result

In [ ]:
result = ask_database(
    "What are the average ROE values by sector?"
)

result

In [ ]:
def generate_answer(question, result):
    result_text = result.to_string(index=False)

    prompt = f"""
Answer the user's financial question using ONLY the database results provided below.

User question:
{question}

Database results:
{result_text}

Instructions:
- Give a clear, concise answer.
- Mention the important numbers.
- Do not invent information.
- If the results contain unusual or extreme values, mention them without changing them.
- Do not provide investment advice unless explicitly asked.
"""

    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return response.output_text

In [ ]:
answer = generate_answer(
    "What are the average ROE values by sector?",
    result
)

print(answer)

In [ ]:
def ask_ai_analyst(question):
    # 1. Generate SQL
    sql = generate_sql(question)

    print("Generated SQL:")
    print(sql)
    print()

    # 2. Validate SQL
    is_safe, message = validate_sql(sql)

    if not is_safe:
        return f"SQL blocked for safety: {message}"

    # 3. Execute SQL
    try:
        result = run_safe_query(sql)
    except Exception as e:
        return f"Database error: {str(e)}"

    # 4. Generate explanation
    answer = generate_answer(question, result)

    return answer

In [ ]:
answer = ask_ai_analyst(
    "Which companies had the highest stock returns between 2020 and 2024?"
)

print(answer)

In [ ]:
answer = ask_ai_analyst(
    "Compare TCS and Infosys based on ROE, net profit margin, debt-to-equity, P/E ratio, and stock return."
)

print(answer)

In [ ]:
answer = ask_ai_analyst(
    "Which companies have ROE above 15%, net profit margin above 10%, P/E below 30, debt-to-equity below 2, and stock return above 50%?"
)

print(answer)

In [ ]:
def analyze_question(question):
    # Generate SQL
    sql = generate_sql(question)

    # Validate SQL
    is_safe, message = validate_sql(sql)

    if not is_safe:
        return {
            "success": False,
            "sql": sql,
            "error": message,
            "data": None
        }

    # Execute SQL
    try:
        result = run_safe_query(sql)

        return {
            "success": True,
            "sql": sql,
            "error": None,
            "data": result
        }

    except Exception as e:
        return {
            "success": False,
            "sql": sql,
            "error": str(e),
            "data": None
        }

In [ ]:
analysis = analyze_question(
    "Compare TCS and Infosys based on ROE and stock return."
)

print("Success:", analysis["success"])
print("\nSQL:")
print(analysis["sql"])
print("\nData:")
display(analysis["data"])

In [ ]:
TOOLS = {
    "get_company_profile": get_company_profile,
    "compare_companies": compare_companies,
    "get_sector_analysis": get_sector_analysis,
    "get_valuation": get_valuation,
    "get_stock_performance": get_stock_performance,
    "run_readonly_sql": run_safe_query
}

print("Available tools:")
for tool in TOOLS:
    print("-", tool)

In [65]:
gemini_tools = [
    {
        "type": "function",
        "name": "get_company_profile",
        "description": "Get financial profile and stock performance for a specific company.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "Company name or company ID, such as TCS or INFY."
                }
            },
            "required": ["company_name"],
            "additionalProperties": False
        }
    },

    {
        "type": "function",
        "name": "compare_companies",
        "description": "Compare two companies using financial and stock performance metrics.",
        "parameters": {
            "type": "object",
            "properties": {
                "company1": {
                    "type": "string",
                    "description": "First company ID, such as TCS."
                },
                "company2": {
                    "type": "string",
                    "description": "Second company ID, such as INFY."
                }
            },
            "required": ["company1", "company2"],
            "additionalProperties": False
        }
    },

    {
        "type": "function",
        "name": "get_sector_analysis",
        "description": "Get financial performance metrics by sector. This tool requires no arguments.",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },

    {
        "type": "function",
        "name": "get_valuation",
        "description": "Get company valuation metrics including P/E and P/B ratios. This tool requires no arguments.",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },

    {
        "type": "function",
        "name": "get_stock_performance",
        "description": "Get historical stock performance and returns for companies. This tool requires no arguments.",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },

    {
        "type": "function",
        "name": "run_readonly_sql",
        "description": "Execute a read-only SQL SELECT query against the financial analytics database. Use this only when the available specialized tools cannot answer the question.",
        "parameters": {
            "type": "object",
            "properties": {
                "sql": {
                    "type": "string",
                    "description": "A read-only SQL SELECT or WITH query."
                }
            },
            "required": ["sql"],
            "additionalProperties": False
        }
    }
]

print("Gemini tools defined successfully.")

Gemini tools defined successfully.


In [ ]:
print(gemini_tools)

In [ ]:
response = client.interactions.create(
    model="gemini-3.6-flash",
    input="Compare TCS and Infosys based on ROE and stock return.",
    tools=gemini_tools
)

for step in response.steps:
    if step.type == "function_call":
        print("Function:", step.name)
        print("Arguments:", step.arguments)

In [ ]:
def execute_tool(tool_name, arguments):
    if tool_name not in TOOLS:
        raise ValueError(f"Unknown tool: {tool_name}")
    
    return TOOLS[tool_name](**arguments)


tool_result = None

for step in response.steps:
    if step.type == "function_call":
        print("Executing:", step.name)
        print("Arguments:", step.arguments)
        
        tool_result = execute_tool(
            step.name,
            step.arguments
        )

display(tool_result)

In [ ]:
print(type(response))
print(response)

In [ ]:
for step in response.steps:
    print("Step type:", step.type)

In [ ]:
function_call = next(
    step for step in response.steps
    if step.type == "function_call"
)

tool_response = client.interactions.create(
    model="gemini-3.6-flash",
    previous_interaction_id=response.id,
    input=[
        {
            "type": "function_result",
            "name": function_call.name,
            "result": tool_result.to_dict(orient="records")
        }
    ]
)

print(tool_response.output_text)

In [ ]:
def ai_tool_analyst(question):
    # Ask Gemini which tool to use
    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=question,
        tools=gemini_tools
    )

    # Check whether Gemini requested a tool
    function_call = next(
        (step for step in response.steps if step.type == "function_call"),
        None
    )

    # If no tool was requested
    if function_call is None:
        return extract_response_text(response)

    # Execute the requested tool
    tool_result = execute_tool(
        function_call.name,
        function_call.arguments
    )

    # Send result back to Gemini
    final_response = client.interactions.create(
        model="gemini-3.6-flash",
        previous_interaction_id=response.id,
        input=[
            {
                "type": "function_result",
                "name": function_call.name,
                "result": tool_result.to_dict(orient="records")
            }
        ]
    )

    return extract_response_text(final_response)

In [ ]:
answer = ai_tool_analyst(
    "What is the financial profile of TCS?"
)

print(answer)

In [ ]:
answer = ai_tool_analyst(
    "Which sectors have the highest average ROE?"
)

print(answer)

In [ ]:
answer = ai_tool_analyst(
    "Compare TCS and Infosys based on ROE, net profit margin, debt-to-equity, P/E ratio and stock return."
)

print(answer)

In [69]:
import inspect

print(inspect.getsource(ai_tool_analyst))

def ai_tool_analyst(question):
    # Ask Gemini which tool to use
    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=question,
        tools=gemini_tools
    )

    # Check whether Gemini requested a tool
    function_call = next(
        (step for step in response.steps if step.type == "function_call"),
        None
    )

    # If no tool was requested
    if function_call is None:
        return response.output_text

    # Execute the requested tool
    tool_result = execute_tool(
        function_call.name,
        function_call.arguments
    )

    # Send result back to Gemini
    final_response = client.interactions.create(
        model="gemini-3.6-flash",
        previous_interaction_id=response.id,
        input=[
            {
                "type": "function_result",
                "name": function_call.name,
                "result": tool_result.to_dict(orient="records")
            }
        ]
    )

    return final_response.outpu